# FOML Project — First CNN Baseline
## AI-generated vs Real Image Detection

**Goal:** Train a small CNN from scratch using our already-fixed dataset split.

### Important rules
- Use the same `split_foml_dataset.zip` that was used for the classical ML experiments.
- Training images are used to fit the model.
- Validation images are used to monitor training and report preliminary results.
- **The test set is deliberately not evaluated in this notebook.** We will use it only after selecting our final classical model and CNN.
- This notebook uses a small, explainable CNN; it does not use pretrained weights.

Input labels are fixed as `real = 0` and `ai_generated = 1`, so a predicted probability above 0.5 means the model predicts **AI-generated**.

## 1. Import the libraries

TensorFlow/Keras builds and trains the CNN. NumPy and pandas handle arrays and results. Matplotlib draws the training curves and confusion matrix. Scikit-learn calculates evaluation metrics.

In [ ]:
import os
import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

print("TensorFlow version:", tf.__version__)
print("GPU available:", len(tf.config.list_physical_devices("GPU")) > 0)

tf.keras.utils.set_random_seed(42)

## 2. Upload and prepare the existing fixed split

Upload `split_foml_dataset.zip`. The cell below copies images into a simple two-class folder structure so Keras can load them. It does not change the labels or split assignments.

In [ ]:
from google.colab import files

uploaded = files.upload()
zip_files = [name for name in uploaded.keys() if name.lower().endswith(".zip")]

if not zip_files:
    raise ValueError("Please upload split_foml_dataset.zip")

preferred = [name for name in zip_files if "split_foml_dataset" in name.lower()]
zip_path = preferred[0] if preferred else zip_files[0]
print("Using:", zip_path)

extract_dir = Path("/content/foml_cnn_source")
data_dir = Path("/content/foml_cnn_data")

if extract_dir.exists():
    shutil.rmtree(extract_dir)
if data_dir.exists():
    shutil.rmtree(data_dir)

extract_dir.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(extract_dir)

# Find the dataset root inside the ZIP.
root_candidates = list(extract_dir.rglob("split_metadata.csv"))
if not root_candidates:
    raise FileNotFoundError("Could not find split_metadata.csv inside the ZIP.")

dataset_root = root_candidates[0].parent
print("Dataset root:", dataset_root)

In [ ]:
# Copy images into train/validation/test -> real/ai_generated.
# We keep test images stored, but do not load or evaluate them in this notebook.
splits = ["train", "validation", "test"]
labels = ["real", "ai_generated"]
image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

for split in splits:
    for label in labels:
        source_folder = dataset_root / split / label
        if not source_folder.exists():
            raise FileNotFoundError(f"Missing expected folder: {source_folder}")

        target_folder = data_dir / split / label
        target_folder.mkdir(parents=True, exist_ok=True)

        image_paths = [
            p for p in source_folder.rglob("*")
            if p.is_file() and p.suffix.lower() in image_extensions
        ]

        for source_path in image_paths:
            # Prefix category folder name to prevent filename clashes.
            category = source_path.parent.name
            target_name = f"{category}_{source_path.name}"
            shutil.copy2(source_path, target_folder / target_name)

# Show counts before training.
count_rows = []
for split in splits:
    for label in labels:
        folder = data_dir / split / label
        count = sum(1 for p in folder.iterdir() if p.is_file())
        count_rows.append({"split": split, "label": label, "count": count})

count_table = pd.DataFrame(count_rows)
display(count_table)

expected = {
    ("train", "real"): 69,
    ("train", "ai_generated"): 69,
    ("validation", "real"): 15,
    ("validation", "ai_generated"): 15,
    ("test", "real"): 15,
    ("test", "ai_generated"): 15,
}
for row in count_rows:
    key = (row["split"], row["label"])
    if key in expected and row["count"] != expected[key]:
        print(f"WARNING: {key} expected {expected[key]}, found {row['count']}")

print("\nImportant: test set remains reserved and will not be evaluated here.")

## 3. Load training and validation data

Images are loaded at 256 × 256 pixels. The class order is explicitly set to `real` then `ai_generated`, making their numeric labels 0 and 1.

In [ ]:
IMG_SIZE = (256, 256)
BATCH_SIZE = 16
SEED = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir / "train",
    class_names=["real", "ai_generated"],
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=True,
    seed=SEED,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir / "validation",
    class_names=["real", "ai_generated"],
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False,
)

print("Class names:", train_ds.class_names)

## 4. Preview a few training images

This is a quick visual check before training. The labels shown are the target labels from our folders.

In [ ]:
plt.figure(figsize=(10, 7))
for images, labels_batch in train_ds.take(1):
    n = min(8, len(images))
    for i in range(n):
        ax = plt.subplot(2, 4, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        label_value = int(labels_batch[i].numpy()[0])
        plt.title("AI-generated" if label_value == 1 else "Real")
        plt.axis("off")
plt.tight_layout()
plt.show()

## 5. Build a small CNN

- **Convolution layers** learn patterns such as edges, textures and shapes.
- **Max pooling** reduces the spatial size of feature maps.
- **Global average pooling** summarizes each feature map without creating a huge dense layer.
- **Dropout** randomly turns off some activations during training to help reduce overfitting.
- **Sigmoid output** produces a score between 0 and 1 for the AI-generated class.

The model is trained from scratch; it uses no pretrained image model.

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.03),
], name="data_augmentation")

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(256, 256, 3)),
    tf.keras.layers.Rescaling(1.0 / 255.0),
    data_augmentation,

    tf.keras.layers.Conv2D(16, (3, 3), activation="relu", padding="same"),
    tf.keras.layers.MaxPooling2D((2, 2)),

    tf.keras.layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    tf.keras.layers.MaxPooling2D((2, 2)),

    tf.keras.layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    tf.keras.layers.MaxPooling2D((2, 2)),

    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.30),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

model.summary()

## 6. Train the CNN

Early stopping monitors validation loss and restores the best observed model weights. Because this dataset is small, do not expect validation metrics to be stable across every random split or run.

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1,
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=[early_stopping],
    verbose=1,
)

## 7. Plot training and validation curves

If training accuracy rises while validation accuracy stalls or falls, or training loss keeps dropping while validation loss increases, this can indicate overfitting.

In [ ]:
history_df = pd.DataFrame(history.history)
display(history_df.tail())

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history_df["accuracy"], label="Training accuracy")
plt.plot(history_df["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("CNN accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_df["loss"], label="Training loss")
plt.plot(history_df["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("CNN loss")
plt.legend()

plt.tight_layout()
plt.savefig("/content/cnn_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

history_df.to_csv("/content/cnn_training_history.csv", index=False)

## 8. Evaluate on the validation set only

The validation set contains 30 images, so each image changes accuracy by about 3.3 percentage points. Treat the results as preliminary. We will not evaluate the test set until the classical ML and CNN choices are finalized.

In [ ]:
# Validation dataset is not shuffled, so labels and predictions remain aligned.
y_true = np.concatenate([y.numpy().reshape(-1) for _, y in val_ds]).astype(int)
y_probability_ai = model.predict(val_ds, verbose=0).reshape(-1)
y_pred = (y_probability_ai >= 0.5).astype(int)

val_accuracy = accuracy_score(y_true, y_pred)
val_precision_ai = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
val_recall_ai = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
val_f1_ai = f1_score(y_true, y_pred, pos_label=1, zero_division=0)

cnn_results = pd.DataFrame([{
    "model": "Small CNN (trained from scratch)",
    "accuracy": val_accuracy,
    "precision_ai": val_precision_ai,
    "recall_ai": val_recall_ai,
    "f1_ai": val_f1_ai,
    "validation_images": len(y_true),
    "decision_threshold": 0.5,
}])

display(cnn_results)
print(classification_report(
    y_true,
    y_pred,
    labels=[0, 1],
    target_names=["Real", "AI-generated"],
    zero_division=0,
))

cnn_results.to_csv("/content/cnn_validation_results.csv", index=False)

## 9. Confusion matrix

Rows show the actual class; columns show the predicted class. Class 0 is Real and class 1 is AI-generated.

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, interpolation="nearest")
ax.set_title("CNN validation confusion matrix")
ax.set_xlabel("Predicted label")
ax.set_ylabel("Actual label")
ax.set_xticks([0, 1], labels=["Real", "AI-generated"])
ax.set_yticks([0, 1], labels=["Real", "AI-generated"])

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center")

fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig("/content/cnn_validation_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Save the model and key results

The model is saved in Keras' `.keras` format. We save the validation results and training curves as well, so the work can later be copied into our GitHub repository. The original dataset is not included in this export.

In [ ]:
model.save("/content/cnn_model.keras")

with open("/content/cnn_class_names.txt", "w") as f:
    f.write("0 = real\n1 = ai_generated\n")

artifact_paths = [
    "/content/cnn_model.keras",
    "/content/cnn_validation_results.csv",
    "/content/cnn_training_history.csv",
    "/content/cnn_training_curves.png",
    "/content/cnn_validation_confusion_matrix.png",
    "/content/cnn_class_names.txt",
]

artifact_zip = "/content/cnn_baseline_artifacts.zip"
with zipfile.ZipFile(artifact_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in artifact_paths:
        if os.path.exists(path):
            zf.write(path, arcname=os.path.basename(path))

print("Saved:", artifact_zip)
print("Model and validation artifacts are ready to download.")
files.download(artifact_zip)

## What to report back

Please share:
1. The final validation results table.
2. The training/validation accuracy and loss curves.
3. The confusion matrix.
4. Whether early stopping finished before 30 epochs.

Do **not** report this validation result as final test accuracy. We will compare it with the classical ML validation results first, then evaluate all selected models once on the reserved 30-image test set.